# Rolling Regression + Fama-MacBeth (v0.4.0)

New features: `rolling_reg()` for time-varying parameter estimation and `fama_macbeth()` for two-pass risk premia estimation with Shanken (1992) correction.

In [1]:
import numpy as np
import polars as pl
import polars_reg as pr

## 1. Rolling Regression

`rolling_reg()` slides a window across the time dimension and runs any polars_reg estimator on each window. It returns a `RollingRegressionResult` with dict-like access by window end-period, stacked coefficient tables, time-series extraction, and optional Altair plotting.

Below we simulate a time series with a **structural break** -- the coefficient on `x` changes from 1.0 to 3.0 at t=100 -- and verify that `rolling_reg()` detects it.

In [2]:
rng = np.random.default_rng(42)
n = 200
x = rng.normal(size=n)
# Structural break: beta changes from 1.0 to 3.0 at t=100
beta = np.where(np.arange(n) < 100, 1.0, 3.0)
y = beta * x + rng.normal(0, 0.5, size=n)

df = pl.DataFrame({"time": range(n), "x": x, "y": y})

result = pr.rolling_reg(pr.ols, "y ~ x", data=df, time="time", window=30, stride=1)
print(f"Windows: {len(result)}, Failed: {len(result.failed)}")
print(result.summary())

Windows: 171, Failed: 0
Rolling Regression: window=30, stride=1, time_col='time'
  171 windows succeeded
  _cons: [-0.1155, 0.3586]
  x: [0.8086, 3.0814]


In [3]:
# coef_series() returns a long-format DataFrame with confidence intervals
series = result.coef_series()
print(series.head(10))

shape: (10, 6)
┌──────┬──────────┬─────────────┬──────────┬───────────┬──────────┐
│ time ┆ variable ┆ coefficient ┆ se       ┆ ci_lower  ┆ ci_upper │
│ ---  ┆ ---      ┆ ---         ┆ ---      ┆ ---       ┆ ---      │
│ i64  ┆ str      ┆ f64         ┆ f64      ┆ f64       ┆ f64      │
╞══════╪══════════╪═════════════╪══════════╪═══════════╪══════════╡
│ 29   ┆ x        ┆ 1.076123    ┆ 0.119457 ┆ 0.831426  ┆ 1.32082  │
│ 29   ┆ _cons    ┆ 0.009149    ┆ 0.091251 ┆ -0.17777  ┆ 0.196067 │
│ 30   ┆ x        ┆ 0.989206    ┆ 0.111457 ┆ 0.760896  ┆ 1.217516 │
│ 30   ┆ _cons    ┆ -0.018746   ┆ 0.095445 ┆ -0.214255 ┆ 0.176763 │
│ 31   ┆ x        ┆ 1.035017    ┆ 0.11033  ┆ 0.809016  ┆ 1.261017 │
│ 31   ┆ _cons    ┆ -0.056364   ┆ 0.09249  ┆ -0.245821 ┆ 0.133094 │
│ 32   ┆ x        ┆ 1.026101    ┆ 0.110962 ┆ 0.798806  ┆ 1.253396 │
│ 32   ┆ _cons    ┆ -0.050631   ┆ 0.092354 ┆ -0.239809 ┆ 0.138548 │
│ 33   ┆ x        ┆ 0.983491    ┆ 0.113564 ┆ 0.750866  ┆ 1.216117 │
│ 33   ┆ _cons    ┆ -0.039799   ┆

In [4]:
# plot_coefs() produces an Altair chart -- the structural break should be visible
# as the x coefficient jumps from ~1.0 to ~3.0 around t=100
result.plot_coefs(variables=["x"])

alt.LayerChart(...)

## 2. Per-Entity Rolling Regression

When `group_by` is provided, `rolling_reg()` runs per-entity rolling windows. Keys become `(entity, window_end)` tuples, and `by_entity()` splits the result into per-entity sub-results.

Here we simulate 5 entities with different true betas and verify that each entity's rolling estimate is stable around its true value.

In [5]:
rng = np.random.default_rng(42)
rows = []
for entity_id in range(5):
    true_beta = 1.0 + entity_id * 0.5  # betas: 1.0, 1.5, 2.0, 2.5, 3.0
    for t in range(60):
        x = rng.normal()
        y = true_beta * x + rng.normal(0, 0.5)
        rows.append({"entity": f"E{entity_id}", "time": t, "x": x, "y": y})

df_panel = pl.DataFrame(rows)
result_grouped = pr.rolling_reg(
    pr.ols, "y ~ x", data=df_panel, time="time", group_by="entity", window=20,
)
print(f"Total windows: {len(result_grouped)}")
print(f"By entity: {list(result_grouped.by_entity().keys())}")
print()

# Show average estimated beta per entity vs true beta
for ent_name, ent_result in sorted(result_grouped.by_entity().items()):
    coefs = [r.coefficients[0] for r in ent_result.values()]
    true_beta = 1.0 + int(ent_name[1]) * 0.5
    print(f"  {ent_name}: true beta={true_beta:.1f}, "
          f"mean estimate={np.mean(coefs):.3f}, "
          f"std={np.std(coefs):.3f}, "
          f"windows={len(ent_result)}")

Total windows: 205
By entity: ['E0', 'E2', 'E4', 'E3', 'E1']

  E0: true beta=1.0, mean estimate=1.076, std=0.050, windows=41
  E1: true beta=1.5, mean estimate=1.551, std=0.106, windows=41
  E2: true beta=2.0, mean estimate=1.933, std=0.076, windows=41
  E3: true beta=2.5, mean estimate=2.708, std=0.053, windows=41
  E4: true beta=3.0, mean estimate=3.167, std=0.075, windows=41


## 3. Fama-MacBeth Two-Pass Estimation

`fama_macbeth()` implements the classic Fama-MacBeth (1973) two-pass procedure for estimating risk premia:

1. **First pass (time-series):** For each entity, regress returns on factors to get factor loadings (betas).
2. **Second pass (cross-sectional):** For each period, regress entity returns on estimated betas to get risk premia (lambdas).

Standard errors are from the time-series variation of lambda estimates. Shanken (1992) correction for generated-regressor bias is applied by default.

We simulate a three-factor model with known risk premia: `mkt=0.5`, `smb=0.3`, `hml=-0.1`.

In [6]:
rng = np.random.default_rng(42)
n_assets, n_periods, n_factors = 50, 120, 3
true_lambdas = np.array([0.5, 0.3, -0.1])

# Factor returns with risk premia
factors = rng.normal(0, 1, (n_periods, n_factors))
factors += true_lambdas  # E[f] = lambda (risk premia)

# Asset betas and returns
betas = rng.normal(1, 0.5, (n_assets, n_factors))
alphas = rng.normal(0, 0.05, n_assets)
epsilon = rng.normal(0, 1, (n_periods, n_assets))
returns = alphas + factors @ betas.T + epsilon

rows = []
for t in range(n_periods):
    for i in range(n_assets):
        rows.append({
            "stock": f"S{i:02d}", "month": t, "ret": returns[t, i],
            "mkt": factors[t, 0], "smb": factors[t, 1], "hml": factors[t, 2],
        })
df_fm = pl.DataFrame(rows)
print(f"Panel: {n_assets} stocks x {n_periods} months = {len(df_fm)} obs")
print(f"True risk premia: mkt={true_lambdas[0]}, smb={true_lambdas[1]}, hml={true_lambdas[2]}")

Panel: 50 stocks x 120 months = 6000 obs
True risk premia: mkt=0.5, smb=0.3, hml=-0.1


In [7]:
# Run Fama-MacBeth with Shanken correction (default)
fm_result = pr.fama_macbeth("ret ~ mkt + smb + hml", data=df_fm, entity="stock", time="month")
print(fm_result.summary())

  Fama-MacBeth (1973) Two-Pass Regression
  N assets:       50        N periods (T_eff):     120
  N obs:        6000        Avg R-squared:      0.4072
                 Mean Lam      FM SE     FM t     FM p        Sh SE     Sh t     Sh p
------------------------------------------------------------------------------------------
mkt                0.5119     0.0898     5.70    <0.01       0.1371     3.73    <0.01
smb                0.2516     0.0946     2.66    <0.01       0.1434     1.75    0.082
hml               -0.1598     0.0868    -1.84    0.068       0.1321    -1.21    0.229
_cons              0.0629     0.0409     1.54    0.126       0.0485     1.30    0.197
  Shanken (1992) correction applied. 'Sh' columns adjust for generated-regressor bias.


In [8]:
# coef_table() returns a Polars DataFrame with FM and Shanken statistics
print("Coefficient table:")
print(fm_result.coef_table())

print(f"\nTrue risk premia: mkt={true_lambdas[0]}, smb={true_lambdas[1]}, hml={true_lambdas[2]}")
print(f"Estimated:        mkt={fm_result.mean_lambda[0]:.4f}, "
      f"smb={fm_result.mean_lambda[1]:.4f}, hml={fm_result.mean_lambda[2]:.4f}")

Coefficient table:
shape: (4, 8)
┌───────┬─────────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┐
│ name  ┆ mean_lambda ┆ fm_se    ┆ fm_t     ┆ fm_p      ┆ shanken_se ┆ shanken_t ┆ shanken_p │
│ ---   ┆ ---         ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---       ┆ ---       │
│ str   ┆ f64         ┆ f64      ┆ f64      ┆ f64       ┆ f64        ┆ f64       ┆ f64       │
╞═══════╪═════════════╪══════════╪══════════╪═══════════╪════════════╪═══════════╪═══════════╡
│ mkt   ┆ 0.511862    ┆ 0.089819 ┆ 5.698844 ┆ 8.9083e-8 ┆ 0.137115   ┆ 3.733073  ┆ 0.000292  │
│ smb   ┆ 0.251627    ┆ 0.09464  ┆ 2.658784 ┆ 0.008922  ┆ 0.143384   ┆ 1.754917  ┆ 0.081847  │
│ hml   ┆ -0.159808   ┆ 0.086753 ┆ -1.84211 ┆ 0.067949  ┆ 0.132147   ┆ -1.209323 ┆ 0.228936  │
│ _cons ┆ 0.062904    ┆ 0.04086  ┆ 1.539492 ┆ 0.126339  ┆ 0.048505   ┆ 1.296868  ┆ 0.197186  │
└───────┴─────────────┴──────────┴──────────┴───────────┴────────────┴───────────┴───────────┘

True risk premia

In [ ]:
# plot_lambdas() shows per-period risk premia with mean + CI bands
# The scatter shows individual period estimates; dashed lines show the FM mean
fm_result.plot_lambdas()

In [9]:
# Access first-pass results (per-entity time-series regressions)
# Without window=, first_pass is a GroupRegressionResult keyed by entity
print(f"First pass type: {type(fm_result.first_pass).__name__}")
print(f"First pass entities: {len(fm_result.first_pass)}")

# Show one entity's estimated betas
s00 = fm_result.first_pass["S00"]
print(f"\nS00 first-pass betas:")
for name, coef, se in zip(s00.names, s00.coefficients, s00.se):
    print(f"  {name:>5}: {coef:+.4f}  (SE={se:.4f})")

First pass type: GroupRegressionResult
First pass entities: 50

S00 first-pass betas:
    mkt: +0.9030  (SE=0.0972)
    smb: +1.1583  (SE=0.0951)
    hml: +1.6781  (SE=0.1024)
  _cons: -0.0269  (SE=0.1100)


## 4. Rolling Fama-MacBeth

When `window` is provided, the first-pass regressions use rolling windows instead of full-sample OLS. This produces time-varying betas, which are then used in the second-pass cross-sectional regressions with a no-look-ahead constraint (each period uses betas from the most recent window ending strictly before that period).

This is the standard approach in empirical asset pricing when betas are expected to change over time.

In [10]:
# Rolling first-pass with 60-period window
fm_rolling = pr.fama_macbeth(
    "ret ~ mkt + smb + hml", data=df_fm, entity="stock", time="month",
    window=60,
)

print("Full-sample FM:")
print(fm_result.coef_table().select(["name", "mean_lambda", "fm_se"]))
print("\nRolling FM (window=60):")
print(fm_rolling.coef_table().select(["name", "mean_lambda", "fm_se"]))
print(f"\nRolling first pass type: {type(fm_rolling.first_pass).__name__}")
print(f"Rolling first pass windows: {len(fm_rolling.first_pass)}")

Full-sample FM:
shape: (4, 3)
┌───────┬─────────────┬──────────┐
│ name  ┆ mean_lambda ┆ fm_se    │
│ ---   ┆ ---         ┆ ---      │
│ str   ┆ f64         ┆ f64      │
╞═══════╪═════════════╪══════════╡
│ mkt   ┆ 0.511862    ┆ 0.089819 │
│ smb   ┆ 0.251627    ┆ 0.09464  │
│ hml   ┆ -0.159808   ┆ 0.086753 │
│ _cons ┆ 0.062904    ┆ 0.04086  │
└───────┴─────────────┴──────────┘

Rolling FM (window=60):
shape: (4, 3)
┌───────┬─────────────┬──────────┐
│ name  ┆ mean_lambda ┆ fm_se    │
│ ---   ┆ ---         ┆ ---      │
│ str   ┆ f64         ┆ f64      │
╞═══════╪═════════════╪══════════╡
│ mkt   ┆ 0.557591    ┆ 0.128928 │
│ smb   ┆ 0.413598    ┆ 0.130204 │
│ hml   ┆ -0.208045   ┆ 0.128308 │
│ _cons ┆ 0.081579    ┆ 0.065514 │
└───────┴─────────────┴──────────┘

Rolling first pass type: RollingRegressionResult
Rolling first pass windows: 3050


In [ ]:
# plot_lambdas() works for rolling FM too — shows time-varying risk premia
fm_rolling.plot_lambdas(variables=["mkt", "smb"])

## 5. regtable Integration

`FamaMacBethResult` duck-types as `RegressionResult`, so it works seamlessly with `regtable()`. This allows side-by-side comparison with pooled OLS or other estimators.

In [11]:
# Compare pooled OLS vs Fama-MacBeth in a single regtable
ols_result = pr.ols("ret ~ mkt + smb + hml", data=df_fm)
pr.regtable(ols_result, fm_result, labels=["Pooled OLS", "Fama-MacBeth"])

,Pooled OLS,Fama-MacBeth
mkt,0.968***,0.5119***
,(52.44),(3.733)
smb,0.8578***,0.2516*
,(47.5),(1.755)
hml,1.133***,-0.1598
,(58.31),(-1.209)
_cons,-0.003168,0.0629
,(-0.1517),(1.297)
N,6000,6000
R²,0.6150,0.4072
